# Implementation: GPT-2 From Scratch

A clean, minimal, from-scratch implementation of a decoder-only GPT-2 language model in pure PyTorch.

**Core Architecture Implemented:**
- Character-level tokenizer and Byte-Pair Encoding (BPE) subword tokenizer
- Token embeddings + learned positional embeddings
- Causal multi-head self-attention with scaled dot-product and causal masking
- Position-wise Feed-Forward Network with GELU activation
- Pre-Layer Normalization and residual stream scaling
- Weight tying between token embedding and final linear projection
- Autoregressive text generation with greedy decoding and temperature sampling
- Empirical experiments (temperature variation, prompt context, loss dynamics, depth ablation)

## 1. Setup & Environment

In [ ]:
import math
import os
import urllib.request
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set seed for reproducibility
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## 2. Tokenization & Dataset Preparation

A character-level discrete tokenizer is used as the primary baseline. The Tiny Shakespeare corpus is split into **90% train** and **10% validation**.

In [ ]:
# Download Tiny Shakespeare
data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(data_url, "tinyshakespeare.txt")

with open("tinyshakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()


class CharTokenizer:
    """Character-level tokenizer."""
    def __init__(self, text: str):
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for i, ch in enumerate(self.chars)}

    def encode(self, text: str) -> list[int]:
        return [self.stoi[ch] for ch in text if ch in self.stoi]

    def decode(self, token_ids: list[int]) -> str:
        return "".join([self.itos[i] for i in token_ids if i in self.itos])


tokenizer = CharTokenizer(text)
chars = tokenizer.chars
vocab_size = tokenizer.vocab_size
encode = tokenizer.encode
decode = tokenizer.decode

print(f"Total characters: {len(text):,}")
print(f"Vocabulary size:  {vocab_size}")

# 90% train, 10% validation split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


def get_batch(split, batch_size=64, block_size=128):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


# Quick check: target y must be shifted by 1 relative to input x
x_sample, y_sample = get_batch("train", batch_size=4, block_size=8)
assert torch.equal(x_sample[0, 1:], y_sample[0, :-1]), "Targets must be shifted by 1!"
print("Batch generation verified.")
print(f"Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,}")


### Tokenizer Exploration: Character-Level vs. Byte-Pair Encoding (BPE)

The character tokenizer maps individual characters to token IDs. Below, a **Byte-Pair Encoding (BPE) subword tokenizer** is implemented from scratch to evaluate sequence compression versus the character-level baseline.

In [ ]:
class BPETokenizer:
    """Byte-Pair Encoding (BPE) subword tokenizer."""
    def __init__(self, vocab_size: int = 256):
        self.target_vocab_size = vocab_size
        self.merges = {}
        self.vocab = {}
        self.stoi = {}
        self.vocab_size = 0

    @staticmethod
    def _get_stats(tokens: list[int]) -> dict:
        counts = {}
        for pair in zip(tokens, tokens[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        return counts

    @staticmethod
    def _merge(tokens: list[int], pair: tuple, new_id: int) -> list[int]:
        new_tokens = []
        i = 0
        n = len(tokens)
        p0, p1 = pair
        while i < n:
            if i < n - 1 and tokens[i] == p0 and tokens[i + 1] == p1:
                new_tokens.append(new_id)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens

    def train(self, text: str):
        chars = sorted(list(set(text)))
        self.vocab = {i: ch for i, ch in enumerate(chars)}
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        tokens = [self.stoi[c] for c in text]

        num_merges = self.target_vocab_size - len(chars)
        for _ in range(num_merges):
            stats = self._get_stats(tokens)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            new_id = len(self.vocab)
            tokens = self._merge(tokens, pair, new_id)
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]

        self.vocab_size = len(self.vocab)

    def encode(self, text: str) -> list[int]:
        tokens = [self.stoi[c] for c in text if c in self.stoi]
        while len(tokens) >= 2:
            pairs = list(zip(tokens, tokens[1:]))
            pair = min(pairs, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            tokens = self._merge(tokens, pair, self.merges[pair])
        return tokens

    def decode(self, token_ids: list[int]) -> str:
        return "".join([self.vocab[i] for i in token_ids if i in self.vocab])


# Compare Character-Level vs. BPE on a sample sentence
sample_sentence = "First Citizen:\nBefore we proceed any further, hear me speak."
bpe = BPETokenizer(vocab_size=160)
bpe.train(text[:30000])

char_ids = encode(sample_sentence)
bpe_ids = bpe.encode(sample_sentence)

print(f"Test Sentence: {repr(sample_sentence)}")
print(f"Character-level tokens: {len(char_ids)} (Vocab: {vocab_size})")
print(f"BPE Subword tokens:     {len(bpe_ids)} (Vocab: {len(bpe.vocab)})")
print(f"Compression factor:     {len(char_ids) / len(bpe_ids):.2f}x")
print(f"Learned Subwords:       {[bpe.vocab[t] for t in bpe_ids[:10]]}")


## 3. GPT-2 Architecture Implemented from Scratch

Every block is built using foundational PyTorch tensor operations:
1. `LayerNorm`: Normalizes across feature dimension from scratch.
2. `CausalSelfAttention`: Q, K, V projections, scaled dot-product ($QK^T / \sqrt{d_k}$), lower-triangular causal mask ($-\infty$), softmax, and value aggregation.
3. `FeedForward`: Position-wise MLP with $4 \times d_{model}$ expansion and GELU.
4. `TransformerBlock`: Pre-LayerNorm formulation ($x + \text{SubLayer}(\text{LN}(x))$) and residual connections.
5. `GPT2`: Embeddings, blocks, weight tying, and autoregressive generation.

In [ ]:
class LayerNorm(nn.Module):
    """Layer Normalization: y = ((x - mean) / sqrt(var + eps)) * gamma + beta"""
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0
        self.d_model = d_model
        self.n_head = n_head
        self.head_dim = d_model // n_head

        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("causal_mask", mask)

    def forward(self, x):
        B, T, C = x.shape  # (B, T, d_model)

        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, n_head, T, head_dim)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))  # (B, n_head, T, T)
        att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        att_weights = self.attn_dropout(F.softmax(att, dim=-1))

        y = att_weights @ v  # (B, n_head, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, d_model)
        return self.resid_dropout(self.out_proj(y))


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.gelu(self.fc1(x))))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln_1 = LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, block_size, dropout)
        self.ln_2 = LayerNorm(d_model)
        self.mlp = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT2(nn.Module):
    def __init__(self, vocab_size, block_size=128, n_layer=4, n_head=4, d_model=128, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_final = LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.token_embedding.weight = self.lm_head.weight

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith("fc2.weight") or pn.endswith("out_proj.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0)

        x = self.drop(self.token_embedding(idx) + self.position_embedding(pos))  # (B, T, d_model)
        for block in self.blocks:
            x = block(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            if temperature == 0.0:
                idx_next = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = float("-inf")
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)
        return idx

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())


## 4. Model Training Pipeline

We train a 4-layer, 4-head, $d_{model}=128$ model (~808K parameters) with AdamW.

In [ ]:
# Model & Training Hyperparameters (Configured for GPU training on Colab)
BLOCK_SIZE = 128
N_LAYER = 4
N_HEAD = 4
D_MODEL = 128
DROPOUT = 0.1
BATCH_SIZE = 64
LEARNING_RATE = 6e-4
MAX_ITERS = 2000
EVAL_INTERVAL = 250
EVAL_ITERS = 50

model = GPT2(
    vocab_size=vocab_size,
    block_size=BLOCK_SIZE,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    d_model=D_MODEL,
    dropout=DROPOUT
).to(device)

print(f"Total trainable parameters: {model.get_num_params():,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)


@torch.no_grad()
def estimate_loss(m):
    m.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split, BATCH_SIZE, BLOCK_SIZE)
            _, loss = m(X, Y)
            losses[k] = loss.item()
        mean_l = losses.mean().item()
        out[split] = mean_l
        out[f"{split}_ppl"] = math.exp(mean_l)
    m.train()
    return out


# History tracking for loss curves
history = {"step": [], "train_loss": [], "val_loss": [], "val_ppl": []}
best_val_loss = float("inf")
os.makedirs("checkpoints", exist_ok=True)
checkpoint_path = "checkpoints/best_model.pt"

print("Starting training...")
for step in range(1, MAX_ITERS + 1):
    xb, yb = get_batch("train", BATCH_SIZE, BLOCK_SIZE)
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step % EVAL_INTERVAL == 0 or step == MAX_ITERS:
        losses = estimate_loss(model)
        history["step"].append(step)
        history["train_loss"].append(losses["train"])
        history["val_loss"].append(losses["val"])
        history["val_ppl"].append(losses["val_ppl"])
        print(f"Step {step:4d}/{MAX_ITERS} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f} | Val PPL: {losses['val_ppl']:.2f}")

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            torch.save({
                "model_state_dict": model.state_dict(),
                "vocab_size": vocab_size,
                "block_size": BLOCK_SIZE,
                "n_layer": N_LAYER,
                "n_head": N_HEAD,
                "d_model": D_MODEL,
                "val_loss": best_val_loss,
                "chars": chars
            }, checkpoint_path)

print(f"Training complete! Best validation loss: {best_val_loss:.4f}")


## 5. Experiment 1: Effect of Temperature on Text Generation

Evaluation of how decoding temperature alters output behavior, ranging from greedy decoding ($T=0$) to high randomness ($T=1.8$).

In [ ]:
prompt = "ROMEO:\n"
encoded = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
temperatures = [0.0, 0.2, 0.5, 0.8, 1.2, 1.8]

print('=' * 65)
print('EXPERIMENT 1: EFFECT OF TEMPERATURE')
print('=' * 65)

for t in temperatures:
    torch.manual_seed(42)
    gen = model.generate(encoded.clone(), max_new_tokens=100, temperature=t)
    label = 'Greedy (T=0.0)' if t == 0.0 else f'T = {t}'
    print(f'\n--- {label} ---')
    print(decode(gen[0].tolist()).strip())
    print('-' * 50)

## 6. Experiment 2: Effect of Prompt Length & Context

Evaluation of how different starting contexts influence model completions.

In [ ]:
prompts = [
    ("Empty / Unconditioned", "\n"),
    ("Short (1 Word)", "KING:"),
    ("Medium (1 Sentence)", "Shall I compare thee to a summer's day?"),
    ("Rich Dialogue Context", "First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\n")
]

print('=' * 65)
print('EXPERIMENT 2: EFFECT OF PROMPT CONTEXT (T=0.8)')
print('=' * 65)

for name, p in prompts:
    enc = torch.tensor([encode(p)], dtype=torch.long, device=device)
    torch.manual_seed(42)
    out = model.generate(enc, max_new_tokens=100, temperature=0.8)
    print(f'\n--- Context: {name} ({len(p)} characters) ---')
    print(decode(out[0].tolist()).strip())
    print('-' * 50)

## 7. Experiment 3: Training & Validation Loss Dynamics

Plotting the learning curves to analyze model convergence and generalization.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(history['step'], history['train_loss'], label='Train Loss', linewidth=2)
plt.plot(history['step'], history['val_loss'], label='Validation Loss', linestyle='--', linewidth=2)
plt.xlabel('Training Steps')
plt.ylabel('Cross-Entropy Loss')
plt.title('Training and Validation Loss Trajectory')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig('loss_dynamics.png', dpi=300)
plt.show()

## 8. Experiment 4: Architectural Ablation Study (Depth: 4 vs 2 Layers)

Evaluation of depth scaling by comparing the **4-Layer Baseline (~808K params)** with a shallower **2-Layer Ablation Model (~412K params)**.

In [ ]:
ablation_model = GPT2(
    vocab_size=vocab_size,
    block_size=BLOCK_SIZE,
    n_layer=2,  # Ablated to 2 layers
    n_head=N_HEAD,
    d_model=D_MODEL,
    dropout=DROPOUT
).to(device)

print(f'Baseline Model (4 layers): {model.get_num_params():,} params')
print(f'Ablated Model  (2 layers): {ablation_model.get_num_params():,} params')

ablation_opt = torch.optim.AdamW(ablation_model.parameters(), lr=LEARNING_RATE)
ablation_history = {'step': [], 'val_loss': []}

ABLATION_STEPS = 1000
print(f'Training 2-layer ablation model for {ABLATION_STEPS} steps...')
for step in range(1, ABLATION_STEPS + 1):
    xb, yb = get_batch('train', BATCH_SIZE, BLOCK_SIZE)
    _, loss = ablation_model(xb, yb)
    ablation_opt.zero_grad(set_to_none=True)
    loss.backward()
    ablation_opt.step()

    if step % 200 == 0 or step == ABLATION_STEPS:
        losses = estimate_loss(ablation_model)
        ablation_history['step'].append(step)
        ablation_history['val_loss'].append(losses['val'])
        print(f'Ablation Step {step:4d}/{ABLATION_STEPS} | Val Loss: {losses["val"]:.4f}')

# Comparative Plot
plt.figure(figsize=(8, 4.5))
baseline_steps = [s for s in history['step'] if s <= ABLATION_STEPS]
baseline_losses = history['val_loss'][:len(baseline_steps)]

plt.plot(baseline_steps, baseline_losses, label='4-Layer Baseline (808K params)', marker='o', linewidth=2)
plt.plot(ablation_history['step'], ablation_history['val_loss'], label='2-Layer Ablated (412K params)', marker='s', linewidth=2)
plt.xlabel('Training Steps')
plt.ylabel('Validation Loss')
plt.title('Ablation Study: Convergence of 4 Layers vs 2 Layers')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig('ablation_comparison.png', dpi=300)
plt.show()